In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pyarrow
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

#Desactivar notacion científica
pd.set_option('display.float_format', '{:.2f}'.format)

In [5]:
Path.cwd()

WindowsPath('g:/Mi unidad/DS4B_Mastery_Edition/proyectos/ecommerce-growth-analytics/notebooks')

In [6]:
df = pd.read_parquet("../data/intermediate/df_clean.parquet")

In [7]:
df.head()

,date,event,product_id,category,price,user_id,user_session
0,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5


#### Creacion variables temporales

In [8]:
# Eliminar zona horaria (+00:00)
df["date_time"] = df["date"].dt.tz_localize(None)

# Crear variables temporales
df["date"] = df["date_time"].dt.normalize()
df["year"] = df["date_time"].dt.year
df["month"] = df["date_time"].dt.month_name().astype("category")
df["day"] = df["date_time"].dt.day
df["day_of_week"] = df["date_time"].dt.day_name().astype("category")
df["hour"] = df["date_time"].dt.hour

In [9]:
df.set_index('date_time', inplace=True)

In [10]:
df.head(1)

,date,event,product_id,category,price,user_id,user_session,year,month,day,day_of_week,hour
date_time,,,,,,,,,,,,
2019-10-01 00:01:46,2019-10-01,view,5843665,1487580005092295511,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,2019,October,1,Tuesday,0


In [11]:
# Creacion variable holidays con dias festivos rusos
import holidays

years = sorted([2019,2020])
f = holidays.RU(years=years)
holiday_dates = pd.Index(pd.to_datetime(list(f.keys())))
df["holiday"] = np.where(df["date"].isin(holiday_dates), 1, 0)

In [12]:
df.holiday.value_counts()

holiday
0    1940833
1     133193
Name: count, dtype: int64

In [ ]:
#variables exógenas temporales de valor para Rusia
df["is_unity_day"] = (df["date"] == "2019-11-04").astype("int8")
df["is_singles_day"] = (df["date"] == "2019-11-11").astype("int8")
df["is_black_friday"] = (df["date"] == "2019-11-29").astype("int8")
df["is_cyber_monday"] = (df["date"] == "2019-12-02").astype("int8")
df["is_new_year_period"] = (df["date"].between("2019-12-31", "2020-01-08")).astype("int8")
df["is_orthodox_christmas"] = (df["date"] == "2020-01-07").astype("int8")
df["is_valentines_day"] = (df["date"] == "2020-02-14").astype("int8")
df["is_defender_day"] = (df["date"] == "2020-02-23").astype("int8")

Reordenar variables dataset

In [25]:
print(vars)

['date', 'event', 'product_id', 'category', 'price', 'user_id', 'user_session', 'year', 'month', 'day', 'day_of_week', 'hour', 'holiday', 'is_unity_day', 'is_singles_day', 'is_black_friday', 'is_cyber_monday', 'is_new_year_period', 'is_orthodox_christmas', 'is_valentines_day', 'is_defender_day']


In [19]:
vars = df.columns.to_list()

In [23]:
order = ['user_id',
         'user_session',
         'event',
         'product_id',
         'category',
         'price']

vars_new = order + [var for var in vars if var not in order]

In [26]:
print(vars_new)

['user_id', 'user_session', 'event', 'product_id', 'category', 'price', 'date', 'year', 'month', 'day', 'day_of_week', 'hour', 'holiday', 'is_unity_day', 'is_singles_day', 'is_black_friday', 'is_cyber_monday', 'is_new_year_period', 'is_orthodox_christmas', 'is_valentines_day', 'is_defender_day']


In [30]:
df = df[vars_new]

In [33]:
df.head(1)

,user_id,user_session,event,product_id,category,price,date,year,month,day,day_of_week,hour,holiday,is_unity_day,is_singles_day,is_black_friday,is_cyber_monday,is_new_year_period,is_orthodox_christmas,is_valentines_day,is_defender_day
date_time,,,,,,,,,,,,,,,,,,,,,
2019-10-01 00:01:46,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,view,5843665,1487580005092295511,9.44,2019-10-01,2019,October,1,Tuesday,0,0,0,0,0,0,0,0,0,0


In [34]:
df.to_parquet("../data/processed/df_processed.parquet", index=True)